[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module8/01-pep8-style.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module8/01-pep8-style.ipynb)

# Module 8 Lesson 1 — PEP 8 Style & Code Formatters

**Module 8: Best Practices & Real-World Python** | Estimated time: 30 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Explain the core PEP 8 guidelines (naming, line length, blank lines, whitespace)
- Use **Black** to auto-format Python source files
- Use **flake8** and **pycodestyle** to lint code and interpret error codes
- Sort imports automatically with **isort**
- Configure tools via `pyproject.toml`
- Understand pre-commit hooks for enforcing style on every commit
- Use **Ruff** as a fast, all-in-one modern linter/formatter

In [ ]:
# Install all style/linting tools used in this lesson
!pip install black flake8 isort pycodestyle ruff --quiet
print("All tools installed.")

## PEP 8 — The Style Guide for Python Code

[PEP 8](https://peps.python.org/pep-0008/) is the official style guide for Python. Consistent style makes code easier to read and maintain across teams.

### Key naming conventions

| Identifier type | Convention | Example |
|---|---|---|
| Variables & functions | `snake_case` | `user_name`, `calculate_total()` |
| Constants | `UPPER_SNAKE_CASE` | `MAX_RETRIES`, `BASE_URL` |
| Classes | `CamelCase` (PascalCase) | `UserProfile`, `HTTPClient` |
| Private attributes | `_leading_underscore` | `_internal_state` |
| Name-mangled attrs | `__double_leading` | `__private_field` |

### Other core rules

- **Line length**: 79 characters (PEP 8) or 88 characters (Black's default — widely accepted)
- **Blank lines**: 2 blank lines around top-level definitions; 1 blank line between methods
- **Whitespace**: spaces around operators, after commas; no spaces before colons/brackets
- **Imports**: one import per line, stdlib → third-party → local (with blank lines between groups)

In [ ]:
# Demonstrate naming conventions in practice
MAX_RETRIES = 3          # constant — UPPER_SNAKE_CASE
BASE_URL = "https://api.example.com"


class UserProfile:       # class — CamelCase
    def __init__(self, first_name: str, last_name: str) -> None:
        self.first_name = first_name
        self.last_name = last_name
        self._access_token: str | None = None  # private — _leading

    def full_name(self) -> str:               # method — snake_case
        return f"{self.first_name} {self.last_name}"

    def update_token(self, new_token: str) -> None:
        self._access_token = new_token


def calculate_total(prices: list[float], tax_rate: float = 0.1) -> float:
    """Return the total price including tax."""
    subtotal = sum(prices)
    return subtotal * (1 + tax_rate)


# Usage
user = UserProfile("Ada", "Lovelace")
print(user.full_name())
print(calculate_total([10.0, 20.0, 5.50]))

## Bad Code vs Good Code — Whitespace & Structure

Let's look at a before/after comparison that shows common PEP 8 violations.

In [ ]:
# BAD CODE — multiple PEP 8 violations
bad_code = '''
import sys,os
import json
from collections import OrderedDict,defaultdict

MyConstant=42

class myClass:
    def __init__(self,x,y):
        self.x=x
        self.y=y
    def getSum( self ):
        return self.x+self.y

def BadFunction(  X,Y,Z  ):
    result=X*Y+Z
    if result>100 :
        print( "big" )
    return result

print(BadFunction(2,3,4))
'''

print("=== BAD CODE (before formatting) ===")
print(bad_code)

In [ ]:
# Write the bad code to a file so we can format it
with open("bad_code.py", "w") as f:
    f.write(bad_code.strip())

print("bad_code.py written.")

## Black — The Uncompromising Code Formatter

Black reformats your code automatically. It is *opinionated* — there are very few configuration options, which is the point: everyone on the team gets the same output.

```bash
# Format a single file
black myfile.py

# Preview changes without writing (--diff)
black --diff myfile.py

# Check only — exit 1 if file would be reformatted (CI use)
black --check myfile.py
```

In [ ]:
# Show what Black would change (--diff mode)
print("=== BLACK DIFF (proposed changes) ===")
!black --diff --color bad_code.py

In [ ]:
# Apply Black formatting
!black bad_code.py

print("\n=== AFTER BLACK ===")
with open("bad_code.py") as f:
    print(f.read())

## flake8 — Linting for Logic & Style Errors

Black handles *formatting*, but flake8 catches *logical and structural* issues Black won't fix.

### Common error codes

| Code | Meaning |
|---|---|
| `E101` | Indentation contains mixed spaces and tabs |
| `E302` | Expected 2 blank lines, found 1 |
| `E501` | Line too long (> 79 characters) |
| `W291` | Trailing whitespace |
| `W503` | Line break before binary operator |
| `F401` | Module imported but unused |
| `F821` | Undefined name |
| `F841` | Local variable assigned but never used |

In [ ]:
%%writefile flake8_demo.py
import os
import sys
import json   # F401 — unused import

VERY_LONG_VARIABLE_NAME_THAT_MAKES_LINE_WAY_TOO_LONG = "this line exceeds the recommended line length for PEP 8"  # E501

def greet(name):
    msg = "Hello, " + name  # F841 — msg assigned but never used if we don't return
    return "Hi " + name


result = greet("World")
print(result)
print(os.getcwd())
print(sys.version)

In [ ]:
# Run flake8 — see the violations
print("=== flake8 output ===")
!flake8 flake8_demo.py

# Run with max-line-length adjusted (Black uses 88)
print("\n=== flake8 with --max-line-length=88 ===")
!flake8 --max-line-length=88 flake8_demo.py

## isort — Automatic Import Sorting

isort organises imports into three sections separated by blank lines:
1. Standard library (`os`, `sys`, `json` …)
2. Third-party packages (`requests`, `numpy` …)
3. Local application imports

Within each section, imports are sorted alphabetically.

In [ ]:
%%writefile unsorted_imports.py
import json
import requests
import os
from collections import defaultdict
import numpy as np
import sys
from pathlib import Path
from typing import Optional, List
import pandas as pd

print("imports loaded")

In [ ]:
# Show what isort would change
print("=== isort --diff ===")
!isort --diff unsorted_imports.py

# Apply it
!isort unsorted_imports.py

print("\n=== AFTER isort ===")
with open("unsorted_imports.py") as f:
    print(f.read())

## pyproject.toml — Centralised Tool Configuration

Instead of maintaining separate `.flake8`, `setup.cfg`, and `.isort.cfg` files, modern Python projects use a single `pyproject.toml`.

In [ ]:
%%writefile pyproject.toml
[tool.black]
line-length = 88
target-version = ["py311"]
include = '\.pyi?$'

[tool.isort]
profile = "black"          # makes isort compatible with Black
line_length = 88
known_first_party = ["myapp"]

[tool.flake8]
max-line-length = 88
extend-ignore = ["E203", "W503"]  # conflicts with Black
exclude = [".git", "__pycache__", "venv", ".venv", "dist", "build"]

[tool.ruff]
line-length = 88
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "W", "I", "UP"]
ignore = ["E203"]

## Pre-commit Hooks — Enforce Style Automatically

Pre-commit hooks run tools automatically before every `git commit`, preventing bad code from entering the repository.

In [ ]:
%%writefile .pre-commit-config.yaml
repos:
  - repo: https://github.com/psf/black
    rev: 23.12.1
    hooks:
      - id: black
        language_version: python3.11

  - repo: https://github.com/pycqa/isort
    rev: 5.13.2
    hooks:
      - id: isort
        args: ["--profile", "black"]

  - repo: https://github.com/pycqa/flake8
    rev: 7.0.0
    hooks:
      - id: flake8
        args: ["--max-line-length=88", "--extend-ignore=E203,W503"]

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.1.9
    hooks:
      - id: ruff
        args: [--fix]

In [ ]:
# Setup instructions (run locally — not in Colab)
setup_instructions = """
To activate pre-commit hooks in your project:

    pip install pre-commit
    pre-commit install          # installs hook into .git/hooks/pre-commit
    pre-commit run --all-files  # run manually on all files once

After that, every 'git commit' will auto-run Black, isort, and flake8.
If any hook fails, the commit is aborted so you can fix the issue first.
"""
print(setup_instructions)

## Ruff — Modern All-in-One Linter & Formatter

Ruff is written in Rust and is 10–100x faster than flake8/pylint. It replaces flake8, isort, pyupgrade, and many other tools.

In [ ]:
# Ruff can lint AND format
print("=== Ruff check (linting) ===")
!ruff check flake8_demo.py

print("\n=== Ruff format --diff (formatting preview) ===")
!ruff format --diff bad_code.py

print("\n=== Ruff version ===")
!ruff --version

In [ ]:
# Ruff can fix many issues automatically with --fix
%%writefile ruff_test.py
import os,sys
from typing import List, Optional  # UP006/UP007 — use list/int | None in 3.10+

def add(x,y):
    z = 10  # F841 unused
    return x+y

result=add(1,2)
print( result )
print(os.getcwd())
print(sys.version)

In [ ]:
print("=== Before ruff --fix ===")
with open("ruff_test.py") as f:
    print(f.read())

!ruff check --fix ruff_test.py
!ruff format ruff_test.py

print("\n=== After ruff --fix + format ===")
with open("ruff_test.py") as f:
    print(f.read())

## Practice Exercises

**Exercise 1 — Name the violations**
Look at the code below and list every PEP 8 violation you can find before running any tool:
```python
class calculate_Area:
    def __init__(self,Width,Height):
        self.Width=Width ; self.Height=Height
    def getArea(self):
        return self.Width*self.Height
MyShape=calculate_Area(5,10)
print(MyShape.getArea())
```

**Exercise 2 — Fix and verify**
Write the code above to a file called `exercise.py`, run `black exercise.py` to auto-format it, then run `flake8 exercise.py` to check for remaining issues. Manually fix any issues flake8 still reports.

**Exercise 3 — Configure Ruff**
Create a `pyproject.toml` that configures Ruff to:
- Target Python 3.11
- Use a line length of 100
- Enable rules `E`, `F`, `W`, `I` (isort), and `UP` (pyupgrade)
- Ignore rule `E501` (line too long)
Then run `ruff check --select ALL your_file.py` on any Python file and explore the output.